# Exercice 1 - Probabilité implicite de défaut

On a une obligation de maturité 4 ans, coupon 4% semestriel, rendement 5% (cap. continue). Taux sans risque 3% (composé). Recovery R = 30%.

On cherche la proba de défaut risque-neutre constante.

In [ ]:
import numpy as np
from scipy.optimize import brentq

In [ ]:
# parametres
nominal = 100
coupon = 0.04
c = coupon / 2 * nominal  # coupon semestriel = 2
y = 0.05
T = 4
r = 0.03
R = 0.30

## Prix de marché

On actualise les flux au rendement continu de 5%.

In [ ]:
dates = np.arange(0.5, T + 0.5, 0.5)  # 0.5, 1, 1.5, ..., 4

prix_marche = sum(c * np.exp(-y*t) for t in dates) + nominal * np.exp(-y*T)
print(f"Prix marché = {prix_marche:.4f}")

In [ ]:
# facteur d'actualisation sans risque (composé annuel)
def DF(t):
    return (1 + r)**(-t)

# prix risk-free pour comparaison
prix_rf = sum(c * DF(t) for t in dates) + nominal * DF(T)
print(f"Prix risk-free = {prix_rf:.4f}")
print(f"Ecart = {prix_rf - prix_marche:.4f}")

## Modélisation du défaut

Le défaut survient en fin d'année j (j=1,...,4) avec proba $(1-p)^{j-1} \cdot p$.

Si défaut en fin d'année j :
- coupons à t < j sont reçus normalement
- coupon à t = j n'est pas reçu (défaut juste avant)
- on récupere R * nominal

Si pas de défaut sur les 4 ans, on reçoit tout.

In [ ]:
def prix_avec_defaut(p):
    """Prix de l'obligation en fonction de la proba de defaut annuelle p"""
    prix = 0
    
    # flux de coupons
    for t in dates:
        annee = int(np.ceil(t))
        if t == int(t):  # coupon de fin d'année -> reçu ssi survie cette année
            surv = (1-p)**int(t)
        else:  # coupon de mi-année -> reçu ssi survie année precedente
            surv = (1-p)**(annee - 1)
        prix += c * surv * DF(t)
    
    # principal si survie totale
    prix += nominal * (1-p)**T * DF(T)
    
    # recouvrement en cas de défaut
    for j in range(1, T+1):
        p_def_j = (1-p)**(j-1) * p
        prix += R * nominal * p_def_j * DF(j)
    
    return prix

In [ ]:
# resolution
p_defaut = brentq(lambda p: prix_avec_defaut(p) - prix_marche, 1e-4, 0.5)
print(f"Probabilité de défaut annuelle p = {p_defaut:.6f} soit {p_defaut*100:.2f}%")

In [ ]:
# verification
print(f"Prix calculé avec p = {prix_avec_defaut(p_defaut):.4f}")
print(f"Prix marché         = {prix_marche:.4f}")

## Verification avec l'approximation de Hull

On sait que $p \approx \frac{y - r_c}{1 - R}$ avec $r_c = \ln(1.03)$

In [ ]:
rc = np.log(1 + r)
p_approx = (y - rc) / (1 - R)
print(f"Approximation : p ≈ {p_approx*100:.2f}%")
print(f"Valeur exacte : p = {p_defaut*100:.2f}%")
print("\nLes deux valeurs sont très proches, ce qui valide le calcul.")

## Interprétation

On trouve une probabilité de défaut risque-neutre d'environ 2.9% par an. C'est cohérent avec le spread de credit de ~200 bps (5% - 3%) qui, divisé par (1-R)=0.7, donne bien environ 2.9%.

Attention : c'est une probabilité risk-neutral, pas historique. Elle incorpore une prime de risque et est donc supérieure à la "vraie" probabilité de défaut.